In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Geometry
L, W, H = 0.5, 0.4, 0.3         # metres
nx, ny, nz = 30, 22, 18
dx = L/(nx-1); dy = W/(ny-1); dz = H/(nz-1)

# Temperatures (Kelvin)
T_heater  = 1273.0               # 1000°C — wall setpoint
T_ambient = 300.0                # 27°C
k_air     = 0.071                # W/mK at ~1000°C

# Multi-layer wall: series thermal resistance R = Σ(t/k)
# Layer 1: Refractory brick   k=1.2  W/mK  t=50mm
# Layer 2: Ceramic fibre      k=0.15 W/mK  t=80mm
# Layer 3: Steel shell        k=50   W/mK  t=3mm
layers = [(1.20, 0.050), (0.15, 0.080), (50.0, 0.003)]
R_wall = sum(t/k for k,t in layers)
U_wall = 1.0 / R_wall

# Linearised radiation (Stefan-Boltzmann)
sigma = 5.67e-8;  eps = 0.85
T_avg  = (T_heater + T_ambient) / 2
h_rad  = eps * sigma * (T_avg**2 + T_ambient**2) * (T_avg + T_ambient)
h_door = 15.0 + 0.3*h_rad        # convection + partial radiation at door

# Biot numbers for Robin BCs
Bi_wall = (U_wall + h_rad) * min(dx,dy,dz) / k_air
Bi_door = h_door * dx / k_air

print(f"R_wall = {R_wall:.4f} m²K/W   U_wall = {U_wall:.2f} W/m²K")
print(f"h_rad  = {h_rad:.2f} W/m²K    h_door = {h_door:.2f} W/m²K")
print(f"Bi_wall= {Bi_wall:.3f}         Bi_door= {Bi_door:.3f}")

R_wall = 0.5751 m²K/W   U_wall = 1.74 W/m²K
h_rad  = 37.10 W/m²K    h_door = 26.13 W/m²K
Bi_wall= 9.433         Bi_door= 6.346


In [ ]:
# FDM coefficients (central difference, ∇²T = 0)
cx = 1/dx**2;  cy = 1/dy**2;  cz = 1/dz**2
denom = 2*(cx + cy + cz)

# Initialise with linear gradient (heater → door)
T = np.zeros((nx,ny,nz))
T_door_est = (T_heater + Bi_door*T_ambient) / (1 + Bi_door)
for i in range(nx):
    T[i,:,:] = T_heater + (T_door_est - T_heater) * i/(nx-1)

def apply_bc(T):
    # 5 heated walls — Dirichlet BC at T_heater
    T[0,  :, :] = T_heater   # back wall  (resistance elements here)
    T[:,  0, :] = T_heater   # left wall
    T[:, -1, :] = T_heater   # right wall
    T[:, :,  0] = T_heater   # floor
    T[:, :, -1] = T_heater   # ceiling
    # Door face — Robin BC (thin insulation + gap infiltration)
    T[-1, :, :] = (T[-2,:,:] + Bi_door*T_ambient) / (1 + Bi_door)
    return T

T = apply_bc(T)

for itr in range(6000):
    T_old = T.copy()

    # Laplace update: ∇²T = 0  (vectorised, all interior nodes at once)
    T[1:-1,1:-1,1:-1] = (
        cx*(T[2:,  1:-1,1:-1] + T[:-2, 1:-1,1:-1]) +
        cy*(T[1:-1,2:,  1:-1] + T[1:-1,:-2, 1:-1]) +
        cz*(T[1:-1,1:-1,2:]   + T[1:-1,1:-1,:-2])
    ) / denom

    T = apply_bc(T)

    res = np.max(np.abs(T - T_old))
    if (itr+1) % 1000 == 0:
        print(f"  iter {itr+1}  res={res:.5f}  T=[{T.min()-273:.0f}, {T.max()-273:.0f}]°C")
    if res < 0.05:
        print(f"Converged at iter {itr+1}  residual={res:.2e}")
        break

Tc = T - 273.15
print(f"\nT_min={Tc.min():.1f}°C   T_max={Tc.max():.1f}°C   T_mean={Tc.mean():.1f}°C")
print(f"Non-uniformity ΔT = {Tc.max()-Tc.min():.1f}°C")

Converged at iter 446  residual=4.96e-02

T_min=46.0°C   T_max=999.9°C   T_mean=887.3°C
Non-uniformity ΔT = 953.8°C


In [ ]:
x_arr = np.linspace(0, L, nx)
y_arr = np.linspace(0, W, ny)
z_arr = np.linspace(0, H, nz)
vmin, vmax = Tc.min(), Tc.max()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#111')

def styled(ax, data2d, xa, ya, title, xl, yl):
    X, Y = np.meshgrid(xa, ya)
    cf = ax.contourf(X, Y, data2d.T, levels=60, cmap='inferno', vmin=vmin, vmax=vmax)
    ax.contour(X, Y, data2d.T, levels=10, colors='white', linewidths=0.3, alpha=0.35)
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel(xl, color='#aaa'); ax.set_ylabel(yl, color='#aaa')
    ax.tick_params(colors='#888'); ax.set_facecolor('#111')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
    ax.annotate(f"max {data2d.max():.0f}°C", xy=(0.97,0.97), xycoords='axes fraction',
                ha='right', va='top', color='yellow', fontsize=9, fontweight='bold')
    ax.annotate(f"min {data2d.min():.0f}°C", xy=(0.97,0.04), xycoords='axes fraction',
                ha='right', va='bottom', color='#88aaff', fontsize=9, fontweight='bold')
    return cf

cf = styled(axes[0], Tc[:,:,nz//2], x_arr, y_arr,
            'Top-Down View (Mid-Z)',    'Length (m)', 'Width (m)')
styled(axes[1],      Tc[:,ny//2,:], x_arr, z_arr,
            'Side View (Mid-Y)',        'Length (m)', 'Height (m)')
styled(axes[2],      Tc[-3,:,:],    y_arr, z_arr,
            'Front — Just Inside Door', 'Width (m)',  'Height (m)')

fig.colorbar(cf, ax=axes.tolist(), label='Temperature (°C)', shrink=0.8)
fig.suptitle("Furnace.exe — 3D Thermal Gradient  |  MET-QUEST'26",
             color='white', fontsize=14, fontweight='bold')
plt.savefig('furnace_output.png', dpi=150, bbox_inches='tight', facecolor='#111')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ─── Geometry & BCs ────────────────────────────────
L, W, H = 0.5, 0.4, 0.3
nx, ny, nz = 30, 22, 18
dx = L/(nx-1); dy = W/(ny-1); dz = H/(nz-1)

T_heater  = 1273.0
T_ambient = 300.0
k_air     = 0.071

layers = [(1.20, 0.050), (0.15, 0.080), (50.0, 0.003)]
R_wall = sum(t/k for k,t in layers)
U_wall = 1.0 / R_wall

sigma = 5.67e-8;  eps = 0.85
T_avg  = (T_heater + T_ambient) / 2
h_rad  = eps * sigma * (T_avg**2 + T_ambient**2) * (T_avg + T_ambient)
h_door = 15.0 + 0.3*h_rad

Bi_wall = (U_wall + h_rad) * min(dx,dy,dz) / k_air
Bi_door = h_door * dx / k_air

# ─── FDM solver ────────────────────────────────────────────────────────────
cx = 1/dx**2;  cy = 1/dy**2;  cz = 1/dz**2
denom = 2*(cx + cy + cz)

T = np.zeros((nx,ny,nz))
T_door_est = (T_heater + Bi_door*T_ambient) / (1 + Bi_door)
for i in range(nx):
    T[i,:,:] = T_heater + (T_door_est - T_heater) * i/(nx-1)

def apply_bc(T):
    T[0,  :, :] = T_heater
    T[:,  0, :] = T_heater
    T[:, -1, :] = T_heater
    T[:, :,  0] = T_heater
    T[:, :, -1] = T_heater
    T[-1, :, :] = (T[-2,:,:] + Bi_door*T_ambient) / (1 + Bi_door)
    return T

T = apply_bc(T)
for itr in range(6000):
    T_old = T.copy()
    T[1:-1,1:-1,1:-1] = (
        cx*(T[2:,  1:-1,1:-1] + T[:-2, 1:-1,1:-1]) +
        cy*(T[1:-1,2:,  1:-1] + T[1:-1,:-2, 1:-1]) +
        cz*(T[1:-1,1:-1,2:]   + T[1:-1,1:-1,:-2])
    ) / denom
    T = apply_bc(T)
    res = np.max(np.abs(T - T_old))
    if res < 0.05:
        break

Tc = T - 273.15
vmin, vmax = Tc.min(), Tc.max()

# ─── Coordinate arrays ─────────────────────────────────────────────────────
x_arr = np.linspace(0, L, nx)
y_arr = np.linspace(0, W, ny)
z_arr = np.linspace(0, H, nz)

# ─── Figure layout ─────────────────────────────────
fig = plt.figure(figsize=(20, 7), facecolor='black')
fig.patch.set_facecolor('black')

CMAP = 'plasma'

# ── helper: draw one 3-D surface slice ────────────────────────────────────
def surf3d(ax, X, Y, Z, C, title, xlabel, ylabel, zlabel, elev=25, azim=-60):
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    facecolors = plt.cm.plasma(norm(C))
    ax.plot_surface(X, Y, Z, facecolors=facecolors,
                    rstride=1, cstride=1, shade=False, alpha=0.95)
    ax.set_facecolor('black')
    ax.set_title(title, color='white', fontsize=10, pad=8)
    ax.set_xlabel(xlabel, color='#aaa', fontsize=8, labelpad=4)
    ax.set_ylabel(ylabel, color='#aaa', fontsize=8, labelpad=4)
    ax.set_zlabel(zlabel, color='#aaa', fontsize=8, labelpad=4)
    ax.tick_params(colors='#888', labelsize=7)
    ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('#333')
    ax.yaxis.pane.set_edgecolor('#333')
    ax.zaxis.pane.set_edgecolor('#333')
    ax.grid(color='#333', linewidth=0.4)
    ax.view_init(elev=elev, azim=azim)

# ────────────────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ix = nx // 2
YY, ZZ = np.meshgrid(y_arr, z_arr, indexing='ij')
data_side = Tc[ix, :, :]
XX_side   = data_side / vmax * L
surf3d(ax1, XX_side, YY, ZZ, data_side,
       f'3D Surface — Side Cut\n(Mid-X slice)',
       'T/Tmax', 'Width (m)', 'Height (m)', elev=20, azim=-50)

ax2 = fig.add_subplot(1, 3, 2, projection='3d')
iz = nz // 2
XX2, YY2 = np.meshgrid(x_arr, y_arr, indexing='ij')
data_top  = Tc[:, :, iz]
surf3d(ax2, XX2, YY2, data_top, data_top,
       f'3D Surface — Top Cut\n(Mid-Z slice)',
       'Length (m)', 'Width (m)', 'T (°C)', elev=25, azim=-55)

sm = plt.cm.ScalarMappable(cmap=CMAP, norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax2, fraction=0.03, pad=0.08, orientation='vertical')
cbar.ax.yaxis.set_tick_params(color='white', labelsize=8)
cbar.set_label('T (°C)', color='white', fontsize=8)
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

ax3 = fig.add_subplot(1, 3, 3, projection='3d')
YY3, ZZ3 = np.meshgrid(y_arr, z_arr, indexing='ij')
data_door = Tc[-3, :, :]
XX_door   = data_door / vmax * (L*0.4)
surf3d(ax3, XX_door, YY3, ZZ3, data_door,
       f'3D Surface — Door Face\n(X = -1)',
       'T/Tmax', 'Width (m)', 'Height (m)', elev=20, azim=50)

fig.suptitle("Furnace.exe — True 3D Visualisation  |  MET-QUEST'26",
             color='#FFD700', fontsize=14, fontweight='bold', y=1.01)

plt.tight_layout(pad=1.5)
plt.savefig('furnace_3d_viz.png', dpi=150, bbox_inches='tight', facecolor='black', edgecolor='none')
print("Saved → furnace_3d_viz.png in the folder.")

# THIS COMMAND FORCES THE IMAGE TO POP UP ON YOUR SCREEN!
plt.show()

Saved → furnace_3d_viz.png in the folder.
